**1. Import thư viện và thiết lập đường dẫn**

In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor

from sklearn.inspection import permutation_importance

# Config đường dẫn thư mục và các thông số 

SEED = 42
TARGET = "quantity_sold"  # biến mục tiêu
LEAK_COLS = ("review_to_sold_ratio",)  # biến gây leakage
THRESHOLD = 2 # chia miền giá trị thành 2 phần <= 2 và > 2
TOP_K_LOW = 40
XGB_N_ITER = 10


TRAIN_CSV = "../../data/processed/train_data_final.csv"
TEST_CSV  = "../../data/processed/test_data_final.csv"

OUT_KNN_ENET = "./knn_enet_predictions.csv"
OUT_KNN_RF   = "./knn_rf_predictions.csv"
OUT_KNN_XGB  = "./knn_xgb_predictions.csv"

**2. Chọn tham số và train mô hình cho cả 2 phần low và high**

### **Pipeline training cho phần low (KNN)**:
1. Train lần 1 để tìm các đặc trưng quan trọng
2. GridSearchCV để tìm tham số tối ưu
3. Train final model với bộ tham số tốt nhất và với các đặc trưng quan trọng
### **Pipeline training cho phần high area**:
#### **ElasticNet**:
1. ElasticNetCV → tìm alpha, l1_ratio
2. Train lại → lấy coef. Chọn feature theo |coef| > threshold
3. Train final model với bộ tham số tốt nhất và các đặc trưng quan trọng
#### **Random Forest**:
1. GridSearchCV (scoring = R²) để tìm các tham số tối ưu
2. Train best Random Forest Model với bộ tham số tốt nhất
3. Lấy các đặc trưng quan trọng sau khi train
#### **XGBoost**:
1. Train lần 1 để tìm các đặc trưng quan trọng
2. RandomizedSearchCV để tìm tham số tối ưu
3. Train final model với bộ tham số tốt nhất và với các đặc trưng quan trọng

In [2]:
# Load dữ liệu và chia tập train test
def load_xy(train_csv=TRAIN_CSV, test_csv=TEST_CSV, target=TARGET, leak_cols=LEAK_COLS):
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)

    X_train = train_df.drop(columns=[target])
    y_train_log = train_df[target]

    X_test  = test_df.drop(columns=[target])
    y_test_log = test_df[target]

    for c in leak_cols:
        if c in X_train.columns: X_train = X_train.drop(columns=[c])
        if c in X_test.columns:  X_test  = X_test.drop(columns=[c])

    return X_train, y_train_log, X_test, y_test_log

# Tính các chỉ số đánh giá 
def compute_metrics(y_true_log, y_pred_log):
    rmse = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    mae  = mean_absolute_error(y_true_log, y_pred_log)
    r2   = r2_score(y_true_log, y_pred_log)

    y_true_real = np.expm1(y_true_log)
    y_pred_real = np.expm1(y_pred_log)

    rmse_real = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae_real  = mean_absolute_error(y_true_real, y_pred_real)
    r2_real   = r2_score(y_true_real, y_pred_real)

    return {
        "RMSE": rmse, "MAE": mae, "R2": r2,
        "RMSE Real": rmse_real, "MAE Real": mae_real, "R2 Real": r2_real
    }

# In các chỉ số đánh giá với quantity_sold dạng log-scale và real-scale
def print_metrics_dict(m, title=""):
    if title:
        print("\n" + "="*90)
        print(title)
        print("="*90)

    print("\nLOG-SCALE:")
    print(f"RMSE: {m['RMSE']:.6f} | MAE: {m['MAE']:.6f} | R2: {m['R2']:.6f}")

    print("\nREAL-SCALE:")
    print(f"RMSE: {m['RMSE Real']:.6f} | MAE: {m['MAE Real']:.6f} | R2: {m['R2 Real']:.6f}")


#  Gate: dự đoán P(Low | X) : xác suất dữ liệu sẽ thuộc vùng <= 2 hay > 2
def train_gate_rf(X_train, y_train_log, threshold=THRESHOLD, seed=SEED):
    # y_train_log đang là log1p(real_quantity_sold)
    y_low = (y_train_log <= threshold).astype(int)

    gate = RandomForestClassifier(
        n_estimators=500,
        random_state=seed,
        n_jobs=-1
    )
    gate.fit(X_train, y_low)
    return gate

# Low area: KNN (Train lần đầu để chọn đặc trưng quan trọng -> GridSearchCV -> train lần cuối)

# feature selection bằng Random Forest importance top-k 
def select_low_features_rf_topk(X_low, y_low_log, top_k=TOP_K_LOW, seed=SEED):
    rf = RandomForestRegressor(n_estimators=400, random_state=seed, n_jobs=-1)
    rf.fit(X_low, y_low_log)
    fi = pd.Series(rf.feature_importances_, index=X_low.columns).sort_values(ascending=False)
    feats = fi.head(min(top_k, len(fi))).index.tolist()
    return feats, fi


def train_low_knn(X_low, y_low_log, top_k=TOP_K_LOW, seed=SEED, perm_repeats=10):
    feats, fi_rf = select_low_features_rf_topk(X_low, y_low_log, top_k=top_k, seed=seed)
    X_low_sel = X_low[feats]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor())
    ])
    grid = {
        "knn__n_neighbors": [3, 5, 7, 11, 15, 21],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1, 2]
    }

    gs = GridSearchCV(pipe, grid, cv=3, scoring="neg_mean_squared_error", n_jobs=-1, verbose=0)
    gs.fit(X_low_sel, y_low_log)
    best = gs.best_estimator_

    perm = permutation_importance(
        best, X_low_sel, y_low_log,
        n_repeats=perm_repeats,
        random_state=seed,
        scoring="neg_mean_squared_error"
    )
    perm_imp = pd.Series(perm.importances_mean, index=X_low_sel.columns).sort_values(ascending=False)

    return best, feats, gs.best_params_, perm_imp


# High area: ElasticNet (CV -> select coef -> train lại)
def train_high_elasticnet(X_high, y_high_log, seed=SEED, coef_threshold=1e-8):
    enet_cv = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.95,0.99],
            alphas=[0.0001,0.001,0.01,0.1,0.5,1,5,10],
            cv=5,
            max_iter=10000,
            random_state=seed
        ))  
    ])
    enet_cv.fit(X_high, y_high_log)
    best_alpha = enet_cv.named_steps["model"].alpha_
    best_l1 = enet_cv.named_steps["model"].l1_ratio_

    tmp = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=10000, random_state=seed))
    ])
    tmp.fit(X_high, y_high_log)
    coef = tmp.named_steps["model"].coef_

    feats = X_high.columns[np.abs(coef) > coef_threshold].tolist()
    if len(feats) == 0:
        feats = X_high.columns.tolist()

    final = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=10000, random_state=seed))
    ])
    final.fit(X_high[feats], y_high_log)

    coef_final = final.named_steps["model"].coef_
    coef_df = pd.DataFrame({"feature": feats, "coef": coef_final})
    coef_df["abs"] = coef_df["coef"].abs()
    coef_df = coef_df.sort_values("abs", ascending=False)

    meta = {"best_alpha": best_alpha, "best_l1_ratio": best_l1}
    return final, feats, meta, coef_df


# High area: Random Forest (GridSearchCV -> train)
def train_high_rf(X_high, y_high_log, seed=SEED):
    rf = RandomForestRegressor(random_state=seed, n_jobs=-1)
    param_grid = {
        "n_estimators": [200, 500],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", "log2"]
    }
    gs = GridSearchCV(rf, param_grid, cv=3, scoring="r2", n_jobs=-1, verbose=0)
    gs.fit(X_high, y_high_log)
    best = gs.best_estimator_

    fi = pd.Series(best.feature_importances_, index=X_high.columns).sort_values(ascending=False)
    return best, list(X_high.columns), gs.best_params_, fi


# High area: XGBoost (train 1 lần lấy đặc trưng quan trọng -> RandomSearch -> train final)
def train_high_xgb(X_high, y_high_log, seed=SEED, n_iter=XGB_N_ITER):
    X_tr, X_val, y_tr, y_val = train_test_split(X_high, y_high_log, test_size=0.2, random_state=seed)

    first = XGBRegressor(
        n_estimators=10000,
        learning_rate=0.1,
        max_depth=6,
        random_state=seed,
        early_stopping_rounds=50,
        n_jobs=-1
    )
    first.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    fi_first = pd.Series(first.feature_importances_, index=X_high.columns).sort_values(ascending=False)
    feats = fi_first.index.tolist()

    param_dist = {
        "n_estimators": [100, 500, 1000, 5000, 10000],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "max_depth": [3, 5, 7, 9],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0]
    }

    rs = RandomizedSearchCV(
        estimator=XGBRegressor(random_state=seed, n_jobs=-1),
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="neg_mean_squared_error",
        cv=3,
        random_state=seed,
        n_jobs=-1,
        verbose=0
    )
    rs.fit(X_tr[feats], y_tr)

    best_params = rs.best_params_
    final = XGBRegressor(**best_params, random_state=seed, n_jobs=-1)
    final.fit(X_high[feats], y_high_log)

    fi_final = pd.Series(final.feature_importances_, index=feats).sort_values(ascending=False)
    return final, feats, best_params, fi_final

**3. KNN (Low) + ElasticNet (High)**

In ra các feature quan trọng trong mình ở cả 2 vùng low và high, in ra các chỉ số đánh giá và xuất file kết quả dự đoán `quantity_sold`.

In [3]:
X_train, y_train_log, X_test, y_test_log = load_xy()

mask_low  = y_train_log <= THRESHOLD
mask_high = y_train_log > THRESHOLD

X_low,  y_low_log  = X_train.loc[mask_low],  y_train_log.loc[mask_low]
X_high, y_high_log = X_train.loc[mask_high], y_train_log.loc[mask_high]

# Gate
gate = train_gate_rf(X_train, y_train_log)
p_low = gate.predict_proba(X_test)[:, 1]

# Low: KNN
low_model, low_feats, low_best_params, low_perm_imp = train_low_knn(X_low, y_low_log)
print("\n[LOW-KNN] Best params:", low_best_params)
print("[LOW-KNN] Selected features (TOP):", low_feats[:20], f"... (total={len(low_feats)})")
print("\n[LOW-KNN] Permutation importance (top 15):")
print(low_perm_imp.head(15).to_string())

# High: ElasticNet
high_model, high_feats, high_meta, high_coef_df = train_high_elasticnet(X_high, y_high_log)
print("\n[HIGH-ElasticNet] Best:", high_meta)
print("[HIGH-ElasticNet] Selected features (TOP):", high_feats[:20], f"... (total={len(high_feats)})")
print("\n[HIGH-ElasticNet] Coef importance (top 15):")
print(high_coef_df.head(15).to_string(index=False))

# Predict Low/High (log-scale)
y_low_pred_log  = low_model.predict(X_test[low_feats])
y_high_pred_log = high_model.predict(X_test[high_feats])

y_low_pred_real  = np.clip(np.expm1(y_low_pred_log), 0, None)
y_high_pred_real = np.clip(np.expm1(y_high_pred_log), 0, None)
y_pred_real = p_low * y_low_pred_real + (1 - p_low) * y_high_pred_real
y_pred_log  = np.log1p(y_pred_real)

# Metrics
m = compute_metrics(y_test_log, y_pred_log)
print_metrics_dict(m, title="HYBRID: KNN(LOW<=2) + ElasticNet(HIGH>2)")

# Save CSV 
pred_df = pd.DataFrame({
    "quantity_sold_ground_truth": y_test_log.values,
    "quantity_sold_predicted": y_pred_log
})
pred_df.to_csv(OUT_KNN_ENET, index=False)
print("\nSaved:", OUT_KNN_ENET)



[LOW-KNN] Best params: {'knn__n_neighbors': 21, 'knn__p': 1, 'knn__weights': 'distance'}
[LOW-KNN] Selected features (TOP): ['rating_average', 'store_review_count', 'original_price', 'reputation_score', 'name_length', 'price_vs_category', 'review_count', 'price', 'name_word_count', 'shop_potential', 'total_follower', 'total_visuals', 'image_count', 'discount_amount', 'category_root_name_Nhà Cửa - Đời Sống', 'category_root_name_Làm Đẹp - Sức Khỏe', 'hot_keyword_count', 'discount_rate', 'category_root_name_Đồ chơi - Mẹ & Bé', 'category_root_name_Thể Thao – Dã Ngoại'] ... (total=40)

[LOW-KNN] Permutation importance (top 15):
price_vs_category     0.121403
name_length           0.120241
total_visuals         0.119946
image_count           0.119845
name_word_count       0.119365
total_follower        0.118149
shop_potential        0.116963
store_review_count    0.116660
price                 0.114444
original_price        0.114316
trust_level           0.098752
discount_rate         0.086

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.615e+00, tolerance: 1.894e+00
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.389e+00, tolerance: 1.910e+00
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, che


[HIGH-ElasticNet] Best: {'best_alpha': np.float64(0.0001), 'best_l1_ratio': np.float64(0.3)}
[HIGH-ElasticNet] Selected features (TOP): ['price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'is_return_policy', 'is_freeship_xtra', 'is_authentic', 'video_count', 'is_brand', 'store_review_count', 'total_follower', 'is_official', 'reputation_score', 'total_visuals', 'has_video', 'discount_amount', 'price_vs_category', 'hot_keyword_count', 'name_length'] ... (total=50)

[HIGH-ElasticNet] Coef importance (top 15):
                                  feature      coef      abs
                             review_count  1.480468 1.480468
                        price_vs_category -0.814674 0.814674
                                    price  0.421678 0.421678
                       store_review_count  0.256614 0.256614
         category_root_name_Nhà Sách Tiki  0.247664 0.247664
   category_root_name_Điện Tử - Điện Lạnh -0.235255 0.235255
    category_root_name_Làm Đẹp - 

**4. KNN (Low) + RandomForest (High)**

In ra các feature quan trọng trong mình ở cả 2 vùng low và high, in ra các chỉ số đánh giá và xuất file kết quả dự đoán `quantity_sold`.

In [7]:
X_train, y_train_log, X_test, y_test_log = load_xy()

mask_low  = y_train_log <= THRESHOLD
mask_high = y_train_log > THRESHOLD

X_low,  y_low_log  = X_train.loc[mask_low],  y_train_log.loc[mask_low]
X_high, y_high_log = X_train.loc[mask_high], y_train_log.loc[mask_high]

# Gate
gate = train_gate_rf(X_train, y_train_log)
p_low = gate.predict_proba(X_test)[:, 1]

# Low: KNN
low_model, low_feats, low_best_params, low_perm_imp = train_low_knn(X_low, y_low_log)
print("\n[LOW-KNN] Best params:", low_best_params)
print("[LOW-KNN] Selected features (TOP):", low_feats[:20], f"... (total={len(low_feats)})")
print("\n[LOW-KNN] Permutation importance (top 15):")
print(low_perm_imp.head(15).to_string())

# High: RF
high_model, high_feats, high_best_params, high_fi = train_high_rf(X_high, y_high_log)
print("\n[HIGH-RF] Best params:", high_best_params)
print("[HIGH-RF] Features used:", high_feats[:20], f"... (total={len(high_feats)})")
print("\n[HIGH-RF] Feature importance (top 15):")
print(high_fi.head(15).to_string())

# Predict low/high (log-scale)
y_low_pred_log  = low_model.predict(X_test[low_feats])
y_high_pred_log = high_model.predict(X_test[high_feats])

y_low_pred_real  = np.clip(np.expm1(y_low_pred_log), 0, None)
y_high_pred_real = np.clip(np.expm1(y_high_pred_log), 0, None)
y_pred_real = p_low * y_low_pred_real + (1 - p_low) * y_high_pred_real
y_pred_log  = np.log1p(y_pred_real)

# Metrics
m = compute_metrics(y_test_log, y_pred_log)
print_metrics_dict(m, title="HYBRID: KNN(LOW<=2) + RandomForest(HIGH>2)")

# Save CSV 
pred_df = pd.DataFrame({
    "quantity_sold_ground_truth": y_test_log.values,
    "quantity_sold_predicted": y_pred_log
})
pred_df.to_csv(OUT_KNN_RF, index=False)
print("\nSaved:", OUT_KNN_RF)



[LOW-KNN] Best params: {'knn__n_neighbors': 21, 'knn__p': 1, 'knn__weights': 'distance'}
[LOW-KNN] Selected features (TOP): ['rating_average', 'store_review_count', 'original_price', 'reputation_score', 'name_length', 'price_vs_category', 'review_count', 'price', 'name_word_count', 'shop_potential', 'total_follower', 'total_visuals', 'image_count', 'discount_amount', 'category_root_name_Nhà Cửa - Đời Sống', 'category_root_name_Làm Đẹp - Sức Khỏe', 'hot_keyword_count', 'discount_rate', 'category_root_name_Đồ chơi - Mẹ & Bé', 'category_root_name_Thể Thao – Dã Ngoại'] ... (total=40)

[LOW-KNN] Permutation importance (top 15):
price_vs_category     0.121403
name_length           0.120241
total_visuals         0.119946
image_count           0.119845
name_word_count       0.119365
total_follower        0.118149
shop_potential        0.116963
store_review_count    0.116660
price                 0.114444
original_price        0.114316
trust_level           0.098752
discount_rate         0.086

**5. KNN (Low) + XGBoost (High)**

In ra các feature quan trọng trong mình ở cả 2 vùng low và high, in ra các chỉ số đánh giá và xuất file kết quả dự đoán `quantity_sold`.

In [5]:
X_train, y_train_log, X_test, y_test_log = load_xy()

mask_low  = y_train_log <= THRESHOLD
mask_high = y_train_log > THRESHOLD

X_low,  y_low_log  = X_train.loc[mask_low],  y_train_log.loc[mask_low]
X_high, y_high_log = X_train.loc[mask_high], y_train_log.loc[mask_high]

# Gate
gate = train_gate_rf(X_train, y_train_log)
p_low = gate.predict_proba(X_test)[:, 1]

# Low: KNN
low_model, low_feats, low_best_params, low_perm_imp = train_low_knn(X_low, y_low_log)
print("\n[LOW-KNN] Best params:", low_best_params)
print("[LOW-KNN] Selected features (TOP):", low_feats[:20], f"... (total={len(low_feats)})")
print("\n[LOW-KNN] Permutation importance (top 15):")
print(low_perm_imp.head(15).to_string())

# High: XGB
high_model, high_feats, high_best_params, high_fi = train_high_xgb(X_high, y_high_log)
print("\n[HIGH-XGB] Best params:", high_best_params)
print("[HIGH-XGB] Features used (TOP):", high_feats[:20], f"... (total={len(high_feats)})")
print("\n[HIGH-XGB] Feature importance (top 15):")
print(high_fi.head(15).to_string())

# Predict low/high (log-scale)
y_low_pred_log  = low_model.predict(X_test[low_feats])
y_high_pred_log = high_model.predict(X_test[high_feats])

y_low_pred_real  = np.clip(np.expm1(y_low_pred_log), 0, None)
y_high_pred_real = np.clip(np.expm1(y_high_pred_log), 0, None)
y_pred_real = p_low * y_low_pred_real + (1 - p_low) * y_high_pred_real
y_pred_log  = np.log1p(y_pred_real)

# Metrics
m = compute_metrics(y_test_log, y_pred_log)
print_metrics_dict(m, title="HYBRID: KNN(LOW<=2) + XGBoost(HIGH>2)")

# Save CSV 
pred_df = pd.DataFrame({
    "quantity_sold_ground_truth": y_test_log.values,
    "quantity_sold_predicted": y_pred_log
})
pred_df.to_csv(OUT_KNN_XGB, index=False)
print("\nSaved:", OUT_KNN_XGB)



[LOW-KNN] Best params: {'knn__n_neighbors': 21, 'knn__p': 1, 'knn__weights': 'distance'}
[LOW-KNN] Selected features (TOP): ['rating_average', 'store_review_count', 'original_price', 'reputation_score', 'name_length', 'price_vs_category', 'review_count', 'price', 'name_word_count', 'shop_potential', 'total_follower', 'total_visuals', 'image_count', 'discount_amount', 'category_root_name_Nhà Cửa - Đời Sống', 'category_root_name_Làm Đẹp - Sức Khỏe', 'hot_keyword_count', 'discount_rate', 'category_root_name_Đồ chơi - Mẹ & Bé', 'category_root_name_Thể Thao – Dã Ngoại'] ... (total=40)

[LOW-KNN] Permutation importance (top 15):
price_vs_category     0.121403
name_length           0.120241
total_visuals         0.119946
image_count           0.119845
name_word_count       0.119365
total_follower        0.118149
shop_potential        0.116963
store_review_count    0.116660
price                 0.114444
original_price        0.114316
trust_level           0.098752
discount_rate         0.086